In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#!pip install torchmetrics

In [3]:
#!unzip /content/drive/MyDrive/SESM/PySESM.zip
!mkdir results_1
!mkdir results_2
!mkdir results_3
!mkdir results_1/plots
!mkdir results_2/plots
!mkdir results_3/plots
!mkdir results_1/stats
!mkdir results_2/stats
!mkdir results_3/stats

mkdir: cannot create directory ‘results_1’: File exists
mkdir: cannot create directory ‘results_2’: File exists
mkdir: cannot create directory ‘results_3’: File exists
mkdir: cannot create directory ‘results_1/plots’: File exists
mkdir: cannot create directory ‘results_2/plots’: File exists
mkdir: cannot create directory ‘results_3/plots’: File exists
mkdir: cannot create directory ‘results_1/stats’: File exists
mkdir: cannot create directory ‘results_2/stats’: File exists
mkdir: cannot create directory ‘results_3/stats’: File exists


In [4]:
#!pip install wandb -qU

In [5]:
import torch
import sys

# Agregar la ruta al sys.path
sys.path.append('/content/drive/MyDrive/SESM')

#import wandb
#from torchmetrics import MeanSquaredError
from sklearn.metrics import mean_squared_error

import numpy as np
from scipy.stats import multivariate_normal

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.base_functions.Function import GaussianFunctions
from PySESM.models.ISTALayer import ISTALayer
from PySESM.utils.loggers import *

from sklearn.model_selection import train_test_split

import pandas as pd
import random
from google.colab import files
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go


In [6]:
SEED=18

## Login de Wandb

In [7]:
#wandb.login()

In [8]:
logger = setup_logger()
logger.setLevel(logging.INFO)

## Definicion de covarianzas no diagnonales

In [9]:

# Non-diagonal covariance
def generate_sigma_tensors():
    """
    Generates non-diagonal covariance tensors for 2D Gaussian distributions.

    Returns:
    tuple: Tuple containing three non-diagonal covariance tensors (sigma1, sigma2, sigma3).
    """
    e0 = torch.tensor([1.0, 1.0], dtype=torch.float32)
    e0 = e0 / e0.norm()

    def generate_sigma(rotation_angle, scaling_factors):
        rotation_matrix = torch.tensor([[np.cos(rotation_angle), -np.sin(rotation_angle)],
                                       [np.sin(rotation_angle), np.cos(rotation_angle)]], dtype=torch.float32)
        e = torch.mm(rotation_matrix, e0.unsqueeze(1))
        E = torch.cat((e0.unsqueeze(1), e), dim=1)
        D = torch.diag(torch.tensor(scaling_factors, dtype=torch.float32))
        return torch.mm(torch.mm(E, D), E.t())

    sigma1 = generate_sigma(np.pi/4, [0.4, 0.1])
    sigma2 = generate_sigma(np.pi/4, [0.05, 0.5])
    sigma3 = generate_sigma(np.pi/4, [0.2, 0.5])

    return sigma1, sigma2, sigma3


In [10]:
sigma1, sigma2, sigma3 = generate_sigma_tensors()

## Definicion de varianzas diagonales

In [11]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

## Crear Gaussianas

In [12]:
# mesh se va a usar para graficacion solamente
# Los datos para entrenar son aleatorios de X,Y, los de testing son con las gaussianas.

def generate_mesh_and_z(sigma1, sigma2, sigma3):
    """
    Generates a mesh grid and corresponding probability density values for a mixture
    of three 2D Gaussian distributions.

    Args:
    sigma1 (torch.Tensor): The covariance tensor for the first Gaussian distribution.
    sigma2 (torch.Tensor): The covariance tensor for the second Gaussian distribution.
    sigma3 (torch.Tensor): The covariance tensor for the third Gaussian distribution.

    Returns:
    tuple: Tuple containing mesh grid arrays (xx, yy) and the corresponding probability
    density values (zz).
    """
    N_points = 50
    xl = -2
    xr = 2

    x = np.linspace(xl, xr, N_points)
    xx, yy = np.meshgrid(x, x)

    X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    zz = (z1 + z2 + z3).reshape(xx.shape)

    return xx, yy, zz


In [13]:

def generate_random_points_and_eval_z(sigma1, sigma2, sigma3):
    """
    Generates random points and evaluates them on a mixture of three 2D Gaussian distributions.

    Args:
    sigma1 (torch.Tensor): The covariance tensor for the first Gaussian distribution.
    sigma2 (torch.Tensor): The covariance tensor for the second Gaussian distribution.
    sigma3 (torch.Tensor): The covariance tensor for the third Gaussian distribution.

    Returns:
    tuple: Tuple containing random points (x, y) and the corresponding probability density values (z).
    """
    N_points = 500  # Number of random points
    xl = -2
    xr = 2

    # Generate random x and y coordinates
    np.random.seed(SEED)
    x = np.random.uniform(xl, xr, N_points)
    y = np.random.uniform(xl, xr, N_points)
    #x_rand, y_rand = np.meshgrid(x, y)

    X_rand = torch.tensor(np.column_stack([x.ravel(), y.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X_rand.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X_rand.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X_rand.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    #zz_rand = (z1 + z2 + z3).reshape(xx.shape)
    zz_rand = z1 + z2 + z3

    return x, y, zz_rand

## Covarianza de gaussianas del experimento
  - 3 Diagonales
  - 2 Diagonales, 1 no diagonal
  - 1 diagonal, 2 no diagonales
  - 3 no diagonales

In [14]:
# 3 diag
xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)
xx_r, yy_r, zz_r = generate_random_points_and_eval_z(sigma1, sigma2, sigma3)

# 2 diag, 1 no diag
xx_1, yy_1, zz_1 = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3)

# 1 diag, 2 no diag
xx_2, yy_2, zz_2 = generate_mesh_and_z(sigma1_d, sigma2, sigma3)
# 3 no diag
xx_3, yy_3, zz_3 = generate_mesh_and_z(sigma1, sigma2, sigma3)

# print("X: ", xx)
# print("Y: ", yy)
# print("Z: ", zz)
print("X: ", xx_r[:10])
print("Y: ", yy_r[:10])
print("Z: ", zz_r[:10])

X:  [ 0.60149697  0.02181349  1.51440588 -1.2726391   1.40893227  1.00054514
  0.66440667  1.95158179 -0.97212631 -1.8867763 ]
Y:  [-0.54656529 -0.8069387  -1.93993463  0.37175716  0.74557406 -0.39847674
 -1.91854694 -0.26061865  0.73075959  0.00426917]
Z:  tensor([2.9800e-02, 5.2544e-03, 8.6374e-04, 3.2853e-02, 9.0486e-02, 9.9205e-01,
        1.0655e-01, 1.8567e-08, 3.9019e-02, 3.9056e-04])


In [15]:

fig = go.Figure(data=[go.Surface(z=zz.numpy(), x=xx, y=yy)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

#fig.show()

# Funcion para los sub-bloques

In [16]:
#Get Squeeze Factor for Output values
#Arg:
#  Y (List): A simple list with numbers (float)
def squeze_factor(Y):
  e_f   = 0.0
  max_y = max(Y)
  if max_y > 1:
    e_f = 1/max_y
  else:
    e_f = 1.0
  return e_f

In [17]:
class SubBlock:
    """
    Represents a sub-block in a 2D grid.

    Attributes:
    - vertices (np.ndarray): The vertices of the sub-block.
    - amplitude (int): The amplitude of the sub-block.
    - samples_inside (list): List of samples inside the sub-block.
    - output_values (list): List of output values.
    - index (list): List of index of the original point X

    Methods:
    - add_point(point): Add a point to the sub-block.
    """
    def __init__(self, amplitude=1, ista_layer=None):
        self.amplitude = amplitude
        self.ista_layer = ista_layer
        self._X  = []
        self._index  = []
        self.predicted_output = []
        self.output_values = []

    def add_point(self, point, y):
        self._X.append(point)
        self.output_values.append(y)


def get_sub_block_vertices(grid_size, row, col):
    """
    Get the vertices of a sub-block in a 2D grid.

    Args:
    - grid_size (int): The number of segments per dimension.
    - row (int): The row index of the sub-block.
    - col (int): The column index of the sub-block.

    Returns:
    np.ndarray: The vertices of the sub-block.
    """
    delta = 1.0 / grid_size
    x0 = col * delta
    x1 = (col + 1) * delta
    y0 = row * delta
    y1 = (row + 1) * delta
    return np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]])


def locate_samples_in_sub_blocks(x_n, y, t, T):
    """
    Locate points in their respective sub-blocks in a 2D grid.

    Args:
    - x_n (np.ndarray): The normalized points between 0 and 1.
    - y (np.narray) : The output values associated with the samples
    - t (np.ndarray): The integer part of the normalized points.
    - T (int): The number of segments per dimension.

    Returns:
    np.ndarray: Array of SubBlock instances representing the sub-blocks.
    """

    sub_blocks = np.empty((T*T), dtype=object)

    for index in range(T*T):
          sub_blocks[index] = SubBlock()

    for i in range(x_n.shape[0]):
        point = x_n[i]
        location = t[i]
        sub_block = sub_blocks[location[0]*T + location[1]]
        sub_block.add_point(point, y[i])

    return sub_blocks

def generate_list_of_subblock(sub_blocks, l_functions):
  """
    Generate a list for all the sub-blocks with there expected squeze factor

    Arg:
      np.ndarray: Array of SubBlock instances representing the sub-blocks.

    Returns:
    list: List of all sub-block with there block.output_values modified.
  """
  list_sub_blocks = []
  for block in sub_blocks:
    print(f"OUTPUT VALUES: {len(block.output_values) }")
    if(len(block.output_values) != 0):
      block.amplitude = squeze_factor(block.output_values)
      block.ista_layer = ISTALayer(l_functions,SEED)
      block.output_values = [value * block.amplitude for value in block.output_values]

      list_sub_blocks.append(block)

  return list_sub_blocks


In [18]:
def data_mapping(X, T):

    #norm_x = (X - X.min()) / ( (X.max() - X.min() ) / (T-1))
    #TODO: separarlo en entrenamiendo e inferencia
    delta = torch.max(X) - torch.min(X)
    eps   = delta*1e-6
    norm_x = ((X - torch.min(X)) / (delta + eps))*T
    t = norm_x.numpy().astype(int)
    x_n = norm_x - t
    return t, x_n


In [19]:
def predict_on_test_set(X_test, model, T, train_sb):
    t_test, x_n_test = data_mapping(X_test, T)

    sorted_predictions = torch.zeros(len(X_test), dtype=torch.float32)  # Tensor para almacenar las predicciones ordenadas

    for row in range(T):
        for col in range(T):
            sub_block_points = x_n_test[(t_test[:, 0] == row) & (t_test[:, 1] == col)]
            indices = np.where((t_test[:, 0] == row) & (t_test[:, 1] == col))[0]

            if len(sub_block_points) > 0:
                X_sub_block = torch.tensor(sub_block_points, dtype=torch.float32)

                try:
                    current_train_block = train_sb[row * T + col]
                    predictions_sub_block = model.predict(X_sub_block, current_train_block.ista_layer)
                    print("CURRENT ISTA on sub block ", current_train_block.ista_layer.h)
                    print("SUM VALUES H ", current_train_block.ista_layer.h.data.sum())
                    sorted_predictions[indices] = predictions_sub_block.float() / current_train_block.amplitude
                except IndexError:
                    # No se hace nada para los bloques dummy
                    pass

    return sorted_predictions


In [20]:
from collections import defaultdict

def count_unique_combinations(T):
    """
    Count the unique combinations in the list generated by the data_mapping function.

    Args:
        T (int): Value T.

    Returns:
        dict: Dictionary with unique combinations as keys and the number of occurrences as values.
    """
    # Crear un diccionario para contar las combinaciones únicas
    conteo_combinaciones = defaultdict(int)

    # Contar las combinaciones únicas en T
    for combinacion in T:
      combinacion_tuple = tuple(combinacion)
      conteo_combinaciones[combinacion_tuple] += 1

    # Imprimir el conteo de combinaciones únicas
    for combinacion, cantidad in conteo_combinaciones.items():
      print(f"Combinación {combinacion}: {cantidad} veces")

# Configuracion y ejecución del modelo

In [26]:
def generate_uniform_sampling(total_points, n_samples=500, min_separation=1):
    """
    Generate uniform sampling indices with a minimum separation criterion.

    Args:
        total_points (int): Total number of data points.
        n_samples (int): Number of samples to generate (default is 500).
        min_separation (int): Minimum separation between selected indices (default is 1).

    Returns:
        list: List of selected indices.

    Example:
        sampled_indices = generate_uniform_sampling(1000, n_samples=200, min_separation=2)
        TODO> HACER ARRAY DE 1 A TOTAL POINTS, HACER RANDOM PERMUTE Y SACAR PRIMEROS 500
        random.permutation

    """
    np.random.seed(SEED)
    selected_indexes = np.random.permutation(total_points)[:n_samples]
    return selected_indexes

def sample_data(x_values, y_values, z_values, sampled_indices):
    """
    Sample data based on selected indices.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values.
        sampled_indices (list): List of indices to sample.

    Returns:
        tuple: Tuple containing sampled X (features) and y (labels).

    Example:
        X, y = sample_data(x_values, y_values, z_values, sampled_indices)
    """

    sampled_x = torch.tensor(x_values[sampled_indices], dtype=torch.float32)
    sampled_y = torch.tensor(y_values[sampled_indices], dtype=torch.float32)
    X = torch.stack((sampled_x, sampled_y), dim=1)
    y = z_values[sampled_indices].clone().detach().to(dtype=torch.float32)
    return X, y

def build_model(n_samples, n_features, l_functions, eig_range, mu_range, vector_range, model_epochs, ista_epochs, ista_alpha, ista_lambd, mu_epochs, rho_epochs, dictionary_alpha, weight_decay):
    """
    Build and configure the SESM (Sparse Evolutionary Structural Modeling) model.

    Args:
    - n_samples (int): Number of training samples.
    - n_features (int): Number of features.
    - l_functions (int): Number of Gaussian functions.

    Returns:
    SESM_Model: An instance of the SESM model.
    """
    gaussian_function = GaussianFunctions(n_features=n_features, n_functions=l_functions,logger=logger, eig_range=eig_range, mu_range=mu_range, vector_range=vector_range,seed=SEED)
    model = SESM_Model(
        n_samples=n_samples,
        psi=gaussian_function,
        seed = SEED,
        model_epochs=model_epochs,
        ista_epochs=ista_epochs,
        ista_alpha=ista_alpha,
        ista_lambd=ista_lambd,
        mu_epochs=mu_epochs,
        rho_epochs=rho_epochs,
        dictionary_alpha=dictionary_alpha,
        weight_decay=weight_decay
    )
    return model

def train_model(model, X, y):
    """
    Train the SESM model using the specified training parameters.

    Args:
    - model (SESM_Model): The SESM model to be trained.
    - X (torch.Tensor): Input features.
    - y (torch.Tensor): Target values.

    Returns:
    None
    """
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model.fit(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )


def plot_surface(x_values, y_values, z_values, Z, sampled_info, hypset, fngroup, iteration, losses_ISTA,losses_Dictionary):
    """
    Plot a surface plot for the original function and the surrogate model.

    Args:
        x_values (array-like): X-axis values.
        y_values (array-like): Y-axis values.
        z_values (array-like): Z-axis values for the original function.
        Z (array-like): Z-axis values for the surrogate model.
        hypset (int): Hypothesis set or configuration identifier.
        fngroup (int): Function group identifier.
        iteration (int): Iteration number.

    Returns:
        None: Saves the plot as an image file.

    Example:
        plot_surface(x_values, y_values, z_values, Z, 3, 1, 1)
    """

    fig = plt.figure(figsize=(15, 10))
    X = x_values.reshape(50, 50)  # Remodelar X a una matriz 2D
    Y = y_values.reshape(50, 50)  # Remodelar Y a una matriz 2D
    Z = Z.clone().reshape(50, 50).detach().numpy()

    x_samples = sampled_info["X_test"][:, 0]
    y_samples = sampled_info["X_test"][:, 1]
    z_samples = sampled_info["y_test"]

    #Total epochs = 6 (2*3 permutaciones) * 16 bloques
    ax1 = fig.add_subplot(231)
    ax1.scatter(range(len(losses_ISTA)), losses_ISTA)
    ax1.set_xlabel('Total epochs')
    ax1.set_ylabel('losses_ISTA')
    ax1.set_title('losses_ISTA vs Total epochs')

    ax5 = fig.add_subplot(232)
    ax5.scatter(range(len(losses_Dictionary)), losses_Dictionary)
    ax5.set_xlabel('Total epochs')
    ax5.set_ylabel('losses_Dictionary')
    ax5.set_title('losses_Dictionary vs Total epochs')

    ax3 = fig.add_subplot(233)
    ax3.scatter(x_samples, y_samples)
    ax3.set_xlabel('X')
    ax3.set_ylabel('Y')
    ax3.set_title('Sampled Data')

    # Ajuste de los límites de los ejes
    ax3.set_xlim([min(x_samples), max(x_samples)])
    ax3.set_ylim([min(y_samples), max(y_samples)])

    ax4 = fig.add_subplot(234, projection='3d')
    ax4.plot_surface(X, Y, z_values.reshape(50, 50), cmap='viridis', alpha=0.9)
    ax4.scatter(x_samples, y_samples, z_samples, c='red')
    ax4.set_xlabel('X')
    ax4.set_ylabel('Y')
    ax4.set_zlabel('Z')
    ax4.set_title('Original Function')

    # Ajuste de los límites de los ejes
    ax4.set_xlim([min(x_samples), max(x_samples)])
    ax4.set_ylim([min(y_samples), max(y_samples)])
    ax4.set_zlim([min(z_samples), max(z_samples)])

    ax2 = fig.add_subplot(212, projection='3d')
    ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
    ax2.scatter(x_samples, y_samples, z_samples, c='red')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Surrogate Model')

    # Ajuste de los límites de los ejes
    ax2.set_xlim([min(x_samples), max(x_samples)])
    ax2.set_ylim([min(y_samples), max(y_samples)])
    ax2.set_zlim([min(z_samples), max(z_samples)])

    filename = f"results_{hypset}/plots/{fngroup}.{iteration}.png"
    plt.tight_layout()
    plt.savefig(filename)
    plt.close(fig)





def evaluate_model(model, x_values, y_values, z_values, sampled_info, hypset, fngroup, iter, T, list_sub_blocks, debug=True):
    """
    Evaluate the SESM model's performance on a dataset.

    Args:
    - model (SESM_Model): The trained SESM model.
    - x_values (Union[np.ndarray, list]): Input feature values along the x-axis.
    - y_values (Union[np.ndarray, list]): Input feature values along the y-axis.
    - z_values (Union[np.ndarray, list]): Target values.
    - hypset (str): Hypothesis set identifier.
    - fngroup (str): Function group identifier (Dataset).
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for evaluation (in minutes) and
    the Mean Squared Error (MSE) value.
    """
    #----------------PREDICTION------------
    x_tensor = torch.tensor(x_values)
    y_tensor = torch.tensor(y_values)
    X_test   =  torch.stack((x_tensor, y_tensor), dim=1)
    t_test, x_n_test = data_mapping(X_test, T)
    Z = predict_on_test_set(X_test, model, T, list_sub_blocks)

    #----------------- ANALYSIS------------
    time = model.time / 60
    #mse = MeanSquaredError()
    mse = mean_squared_error(Z.clone().detach(), z_values)
    #mse_value = mse.compute()

    if debug:
        plot_surface(x_values, y_values, z_values, Z, sampled_info, hypset, fngroup, iter, model.losses_ISTA, model.losses_Dictionary)

        df = pd.DataFrame({
            'Loss_ISTA': model.losses_ISTA,
            'Loss_Dictionary': model.losses_Dictionary
        })

        df.to_csv(f'results_{hypset}/stats/{iter}.csv', index=False)

        """for loss in model.losses:
          wandb.log({"loss": loss})"""



    return time, mse

def run_experiment(_x_train, _y_train, _z_train, _x_test, _y_test, _z_test, hyperparams, fngroup, iter, debug=True):
    """
    Run a complete SESM experiment, including data sampling, model training, and evaluation.

    Args:
    - _x (np.ndarray): Input feature values along the x-axis.
    - _y (np.ndarray): Input feature values along the y-axis.
    - _z (np.ndarray): Target values.
    - hyperparams (dict): Hyperparameters for the experiment.
    - fngroup (str): Function group identifier.
    - iter (int): Iteration number.
    - debug (bool): Whether to generate and save debugging plots (default is True).

    Returns:
    Tuple[float, float]: Tuple containing time taken for the experiment (in minutes)
    and the Mean Squared Error (MSE) value.
    """
    x_values = _x_train.ravel()
    y_values = _y_train.ravel()
    z_values = _z_train.ravel()

    x_test = _x_test.ravel()
    y_test = _y_test.ravel()
    z_test = _z_test.ravel()

    total_points = len(x_values)

    sampled_indices = generate_uniform_sampling(total_points, n_samples=hyperparams["n_samples"])
    X, y = sample_data(x_values, y_values, z_values, sampled_indices)

    sampled_info = { "X_test": X, "y_test": y}
    #t int y x_n float
    t, x_n = data_mapping(X, hyperparams["T"])
    count_unique_combinations(t)

    sub_blocks = locate_samples_in_sub_blocks(x_n, y, t, hyperparams["T"])
    list_sub_blocks = generate_list_of_subblock(sub_blocks, hyperparams["l_functions"])

    n_samples=hyperparams["n_samples"]
    n_features=hyperparams["n_features"]
    l_functions=hyperparams["l_functions"]
    eig_range=hyperparams["eig_range"]
    mu_range=hyperparams["mu_range"]
    vector_range=hyperparams["vector_range"]

    model_epochs = hyperparams["m_epochs"]
    ista_epochs = hyperparams["h_epochs"]
    rho_epochs = hyperparams["rho_epochs"]
    mu_epochs = hyperparams["mu_epochs"]

    ista_alpha = hyperparams["ista_alpha"]
    ista_lambd = hyperparams["ista_lambd"]
    dictionary_alpha = hyperparams["dictionary_alpha"]
    weight_decay = hyperparams["weight_decay"]
    permutation_times = hyperparams["permutation_times"]

    model = build_model(n_samples, n_features, l_functions, eig_range, mu_range, vector_range, model_epochs, ista_epochs, ista_alpha, ista_lambd, mu_epochs, rho_epochs, dictionary_alpha, weight_decay)



    # Iterar sobre list_sub_blocks con diferentes permutaciones
    for _ in range(permutation_times):
      selected_indexes = np.random.permutation(hyperparams["T"]**2)
      permuted_list_sub_blocks = [list_sub_blocks[i] for i in selected_indexes]
      for block in permuted_list_sub_blocks:
         y = torch.tensor(block.output_values, dtype=torch.float32)
         X = torch.tensor(np.array(block._X), dtype=torch.float32)
         model.ista_layer = block.ista_layer
         train_model(model, X, y)

    print("model.losses_ISTA: ",len(model.losses_ISTA))
    print("model.losses_Dictionary: ",len(model.losses_Dictionary))

    time, mse_value = evaluate_model(model, x_test, y_test, z_test, sampled_info, hyperparams["hyp_set"],  fngroup, iter, hyperparams["T"], list_sub_blocks, debug)
    return time, mse_value

In [22]:

def plot_stats(directory, num_files):
    """
    Plot statistics for loss values from multiple CSV files.

    Args:
        directory (str): The directory containing CSV files.
        num_files (int): The number of CSV files to process.

    Returns:
        None: Displays an interactive plot and saves an HTML file.

    Note:
        Assumes that each CSV file contains loss values for the same number of epochs.

    """
    fig = px.scatter(title='Loss analysis')
    m_epochs_losses = []

    for i in range(num_files):
        file_path = f"{directory}/stats/{i}.csv"

        df = pd.read_csv(file_path)
        m_epochs_losses.append(df)

    merged_losses = pd.concat(m_epochs_losses, axis=1)

    # Compute mean, std, min, and max for each row
    summary_df = pd.DataFrame({
        'Mean': merged_losses.mean(axis=1),
        'Std': merged_losses.std(axis=1),
        'Min': merged_losses.min(axis=1),
        'Max': merged_losses.max(axis=1)
    })

    summary_df.to_csv(f'{directory}/stats/processed.csv', index=False)

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Mean'],
        mode='lines+markers',
        error_y=dict(type='data', array=summary_df['Std']),
        name='Mean'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Max'],
        mode='markers',
        name='Max'
    )

    fig.add_scatter(
        x=summary_df.index,
        y=summary_df['Min'],
        mode='markers',
        name='Min'
    )

    fig.update_layout(
        xaxis_title='m_epochs',
        yaxis_title='Loss',
        legend_title='Legend',
        showlegend=True
    )
    fig.show()
    fig.write_html(f"interactive_plot-{directory}.html")

In [23]:
import csv

def save_results(data, fngroup):
  # Compute Mean and Std for executio
  times = [item[1] for item in data]
  mse_values = [item[2] for item in data]

  average_time = np.mean(times)
  std_time = np.std(times)
  average_mse = np.mean(mse_values)
  std_mse = np.std(mse_values)

  with open(f"results_{fngroup}.csv", mode="w", newline="") as file:
      writer = csv.writer(file)
      writer.writerow(["Repetion", "Time (min)", "MSE"])
      writer.writerows(data)
      writer.writerow(["Mean", average_time, average_mse])
      writer.writerow(["Std", std_time, std_mse])


# Nomenclatura de experimentos

<Set de hiperparámetros>. <Set de datos (conjunto de gaussianas)>.<Número de repetición del experimento>


## Set de Hiperparámetros
|  Hiperparámetro | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| n_samples       | 50            | 100           | 500           |
| n_features      | 2             | 2             | 2             |
| l_functions     | 20            | 6             | 10            |
| ista_alpha      | 0.06          | 0.0125        | 0.0125        |
| ista_lambd      | 0.005         | 0.001         | 0.001         |
| dictionary_alpha| 0.06          | 0.0125        | 0.0125        |
| m_epochs        | 25            | 500           | 300           |
| dict_epochs     | 800           | 20            | 60            |
| h_epochs        | 1000          | 50            | 100           |


### Set de datos

|     Set      | Exp 1.x.x     | Exp 2.x.x     | Exp 3.x.x     |
|-----------------|---------------|---------------|---------------|
| Gaussianas 1    | Exp 1.1.x     | Exp 2.1.x     | Exp 3.1.x     |
| Gaussianas 2    | Exp 1.2.x     | Exp 2.2.x     | Exp 3.2.x     |
| Gaussianas 3    | Exp 1.3.x     | Exp 2.3.x     | Exp 3.3.x     |
| Gaussianas 4    | Exp 1.4.x     | Exp 2.4.x     | Exp 3.4.x     |


In [24]:
N_iter = 1 #
experiment_3 = {
      "hyp_set": 1,
      "n_samples"	: 500,
      "n_features" : 2,
      "l_functions":  100,
      "eig_range": [1e0, 1e1],
      "mu_range": [-2, 2],
      "vector_range": [1e0, 1e1],
      "ista_alpha"	: 0.05502,
      "ista_lambd"	 : 0.01007,
      "dictionary_alpha": 0.08928,
      "m_epochs" : 2,
      "rho_epochs": 60,
      "mu_epochs": 60,
      "dict_epochs": 2,
      "h_epochs": 2,
      "T": 4,
      "weight_decay": 0.004875,
      "permutation_times": 3
}
#1, 2, 3 --> m_epochs


# Correr experimentos

In [27]:
data = []
for i in range(N_iter):
    """wandb.init(
        # Set the project where this run will be logged
        project="SESM-2D-GaussianFunction",
        # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
        name=f"12-11-2023-experiments-{i}",
        # Track hyperparameters and run metadata
        config = experiment_3
        )
    """
    time, mse = run_experiment(xx_r,yy_r,zz_r,xx,yy,zz,experiment_3, fngroup=1, iter=i)
    data.append((i, time, mse.item()))
#wandb.finish()
save_results(data=data, fngroup=1)

INFO:logger:Shape of Mu: torch.Size([2, 100])
INFO:logger:Shape of Rho: torch.Size([3, 100])
INFO:logger:Shape of Theta: torch.Size([5, 100])


Combinación (2, 3): 34 veces
Combinación (2, 2): 23 veces
Combinación (2, 1): 49 veces
Combinación (3, 0): 35 veces
Combinación (0, 3): 31 veces
Combinación (1, 2): 38 veces
Combinación (3, 2): 31 veces
Combinación (1, 3): 24 veces
Combinación (0, 0): 34 veces
Combinación (3, 3): 29 veces
Combinación (2, 0): 27 veces
Combinación (1, 0): 28 veces
Combinación (0, 1): 29 veces
Combinación (1, 1): 31 veces
Combinación (0, 2): 26 veces
Combinación (3, 1): 31 veces
OUTPUT VALUES: 34
OUTPUT VALUES: 29
OUTPUT VALUES: 26
OUTPUT VALUES: 31
OUTPUT VALUES: 28
OUTPUT VALUES: 31
OUTPUT VALUES: 38
OUTPUT VALUES: 24
OUTPUT VALUES: 27
OUTPUT VALUES: 49
OUTPUT VALUES: 23
OUTPUT VALUES: 34
OUTPUT VALUES: 35
OUTPUT VALUES: 31
OUTPUT VALUES: 31
OUTPUT VALUES: 29
Epoch 1 Loss_ISTA: 0.046868037432432175 Loss_Dictionary: 0.05807192251086235 

Epoch 2 Loss_ISTA: 0.03335050120949745 Loss_Dictionary: 0.038475800305604935 

Epoch 1 Loss_ISTA: 0.05405453220009804 Loss_Dictionary: 0.057577576488256454 

Epoch 2 Los

<ipython-input-19-e6a96050488e>:12: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [ ]:
data_1 = []

for i in range(N_iter):
  """wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_1,yy_1,zz_1,
                            experiment_3, fngroup=2, iter=i)
  data_1.append((i, time, mse.item()))

save_results(data=data_1, fngroup=2)

In [ ]:
#Unit test de la inicialización únicamente
stats_dir = f'results_{experiment_3["hyp_set"]}'
plot_stats(stats_dir, N_iter)

In [ ]:
data_2 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_2,yy_2,zz_2,
                            experiment_3,fngroup=3, iter=i)
  data_2.append((i, time, mse.item()))

save_results(data=data_2, fngroup=3)

In [ ]:
data_3 = []
for i in range(N_iter):
  """  wandb.init(
    # Set the project where this run will be logged
    project="SESM-2D-GaussianFunction",
    # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
    name=f"12-11-2023-experiments-{i}",
    # Track hyperparameters and run metadata
    config = experiment_3
    )
"""
  time, mse = run_experiment(xx_3,yy_3,zz_3,
                            experiment_3,fngroup=4, iter=i)
  data_3.append((i, time, mse.item()))

save_results(data=data_3, fngroup=4)

In [ ]:
files.download('results_1.csv')
!zip -r results_1.zip results_1/
files.download('results_1.zip')

In [ ]:
#files.download('results_2.csv')
!zip -r results_4.zip results_4/
files.download('results_4.zip')

In [ ]:
# files.download('results_3.csv')
!zip -r results_3.zip results_3/
files.download('results_3.zip')